# Assemble variants around cyp51A\n\nThis notebook extracts the non-synonymous mutations and tandem repeats in the promoter region of cyp51A, combining them into a single GFF3 track.

In [ ]:
import gxy
import gzip
import re
import os

# 1. Read GTF to find coordinates of cyp51A (Afu4g06890)
print("Fetching GTF...")
gtf_paths = await gxy.get("GCF_000002655.1_ASM265v1.ncbiRefSeq.gtf", identifier_type="name")
gtf_path = gtf_paths[0] if isinstance(gtf_paths, list) else gtf_paths

cyp51a_chr = "NC_007197.1"
cyp51a_start = 1777374
cyp51a_end = 1781821

# Let's dynamically find it just in case
try:
    with gzip.open(gtf_path, 'rt') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split('\t')
            if len(parts) > 8 and parts[2] == 'gene' and 'Afu4g06890' in parts[8]:
                cyp51a_chr = parts[0]
                cyp51a_start = int(parts[3])
                cyp51a_end = int(parts[4])
                break
except OSError:
    with open(gtf_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split('\t')
            if len(parts) > 8 and parts[2] == 'gene' and 'Afu4g06890' in parts[8]:
                cyp51a_chr = parts[0]
                cyp51a_start = int(parts[3])
                cyp51a_end = int(parts[4])
                break

print(f"cyp51A found at {cyp51a_chr}:{cyp51a_start}-{cyp51a_end}")

# 2. Extract SNPeff calls
print("Fetching SNPeff VCFs...")
all_snpeff = await gxy.get(".*SnpEff eff: on dataset.*", identifier_type="regex")
if not isinstance(all_snpeff, list): all_snpeff = [all_snpeff]

gff_lines = ["##gff-version 3\n"]

for p in all_snpeff:
    is_vcf = False
    with open(p, 'r') as f:
        head = f.read(50)
        if "##fileformat=VCF" in head:
            is_vcf = True
            
    if not is_vcf: continue
    
    sample_id = os.path.basename(p)
    with open(p, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split('\t')
            chrom = parts[0]
            pos = int(parts[1])
            ref = parts[3]
            alt = parts[4]
            info = parts[7]
            
            if chrom == cyp51a_chr and cyp51a_start <= pos <= cyp51a_end:
                if 'missense_variant' in info or 'nonsynonymous' in info.lower():
                    attr = f"ID=SNP_{sample_id}_{pos};Name={ref}{pos}{alt};Note=Nonsynonymous_SNP"
                    gff_lines.append(f"{chrom}\tSnpEff\tSNP\t{pos}\t{pos}\t.\t.\t.\t{attr}\n")

# 3. Extract etandem calls
print("Fetching etandem outputs...")
all_etandem = await gxy.get(".*etandem on dataset.*", identifier_type="regex")
if not isinstance(all_etandem, list): all_etandem = [all_etandem]

global_offset = 1777374 - 1

for p in all_etandem:
    is_gff = False
    with open(p, 'r') as f:
        head = f.read(50)
        if "##gff-version" in head:
            is_gff = True
            
    if not is_gff: continue
    
    sample_id = os.path.basename(p)
    with open(p, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split('\t')
            if len(parts) >= 9:
                local_start = int(parts[3])
                local_end = int(parts[4])
                
                global_start = global_offset + local_start
                global_end = global_offset + local_end
                
                parts[0] = cyp51a_chr
                parts[3] = str(global_start)
                parts[4] = str(global_end)
                parts[8] = f"Sample={sample_id};" + parts[8].strip()
                
                gff_lines.append("\t".join(parts) + "\n")

# 4. Write to GFF3 and upload
out_file = "cyp51a_variants.gff3"
with open(out_file, "w") as f:
    f.writelines(gff_lines)

print("Uploading GFF3 to Galaxy...")
await gxy.put(out_file, output="cyp51A Combined Variants", ext="gff3")
print("Done!")
